In [16]:
import pandas as pd
import numpy as np
import joblib

### Funciones

In [17]:
def clean_data(df):

    df = df.copy()

    cols_drop = [
        "reservation_status",
        "reservation_status_date",
        "agent",
        "company",
        "country"
    ]

    df.drop(
        columns=cols_drop,
        inplace=True,
        errors="ignore"
    )

    mask_people = (
        (df["adults"] == 0)
        &
        (df["children"].fillna(0) == 0)
        &
        (df["babies"] == 0)
    )

    df = df[~mask_people]

    df.dropna(
        subset=["children"],
        inplace=True
    )

    df["children"] = df["children"].astype(int)

    df = df[df["children"] != 10]

    df = df[df["adr"] >= 0]

    p99 = df["adr"].quantile(0.99)

    df["adr"] = df["adr"].clip(
        upper=p99
    )

    return df

In [18]:
def feature_engineering(df):

    df = df.copy()

    df["total_nights"] = (
        df["stays_in_week_nights"]
        +
        df["stays_in_weekend_nights"]
    )

    df["total_guests"] = (
        df["adults"]
        +
        df["children"]
        +
        df["babies"]
    )

    high_season = [
        "July",
        "August"
    ]

    df["is_high_season"] = (
        df["arrival_date_month"]
        .isin(high_season)
        .astype(int)
    )

    df["adr_log"] = np.log1p(
        df["adr"]
    )

    df["lead_time_log"] = np.log1p(
        df["lead_time"]
    )

    return df

In [19]:
model = joblib.load(
    "../models/final_pipeline.pkl"
)

In [20]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/01-hotel_bookings.csv"
)

print(df.shape)

df.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,03-07-15


In [21]:
df = clean_data(df)

df = feature_engineering(df)

In [22]:
X = df.drop("is_canceled", axis=1)

y = df["is_canceled"]

In [23]:
y_pred = model.predict(X)

y_proba = model.predict_proba(X)[:,1]